# 01 - Hello Qubit: Your First Circuit with QONTOS

> **Environment:** Works offline with `qontos` SDK only. No quantum hardware or API key required.
> **Status:** Beginner | Estimated time: 5 minutes

This notebook walks you through the fundamental QONTOS workflow:

1. Write a quantum circuit in **OpenQASM 2.0**
2. **Normalize** it with `CircuitNormalizer` to produce a provider-agnostic `CircuitIR`
3. Inspect the **metadata** that QONTOS extracts automatically

The normalizer is the entry point to the entire QONTOS pipeline. Every circuit
passes through it before partitioning, scheduling, or execution.

## Prerequisites

```bash
pip install git+https://github.com/qontos/qontos.git@v0.2.0
```

In [ ]:
from qontos.circuit import CircuitNormalizer
from qontos.circuit.metadata import extract_metadata

## Define a Bell State Circuit

A Bell state is the simplest entangled state: apply a Hadamard gate to qubit 0,
then a CNOT from qubit 0 to qubit 1. The result is the state
$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$.

We write it in OpenQASM 2.0, the most widely supported circuit format.

In [ ]:
bell_qasm = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0],q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];
"""

print(bell_qasm)

## Normalize the Circuit

The `CircuitNormalizer` accepts any supported format (`openqasm`, `qiskit`, `pennylane`)
and produces a **CircuitIR** -- the canonical internal representation that all
downstream QONTOS services consume.

In [ ]:
normalizer = CircuitNormalizer()
circuit_ir = normalizer.normalize(input_type="openqasm", source=bell_qasm)

print(f"Type          : {type(circuit_ir).__name__}")
print(f"Num qubits    : {circuit_ir.num_qubits}")
print(f"Classical bits: {circuit_ir.num_clbits}")
print(f"Circuit depth : {circuit_ir.depth}")
print(f"Gate count    : {circuit_ir.gate_count}")
print(f"Source type   : {circuit_ir.source_type}")

## Circuit Hash

QONTOS computes a **deterministic SHA-256 hash** of the circuit's canonical form.
This hash uniquely identifies the circuit regardless of formatting differences,
and is used for deduplication, caching, and integrity verification throughout
the pipeline.

In [ ]:
print(f"Circuit hash: {circuit_ir.circuit_hash}")

# Normalizing the same circuit again produces the same hash
circuit_ir_2 = normalizer.normalize(input_type="openqasm", source=bell_qasm)
assert circuit_ir.circuit_hash == circuit_ir_2.circuit_hash, "Hashes must be deterministic!"
print("Deterministic: same circuit -> same hash (verified)")

## Extract Metadata

QONTOS automatically extracts rich metadata from every circuit. This metadata
drives intelligent decisions in the partitioner, scheduler, and backend router.

In [ ]:
metadata = extract_metadata(circuit_ir)

for key, value in metadata.items():
    print(f"  {key:.<30} {value}")

## Inspect the Gate List

The `CircuitIR` stores every gate as a `GateOperation` with name, qubit indices,
and optional parameters. This is the provider-agnostic representation that
executors translate into their native format.

In [ ]:
print(f"{'Gate':<12} {'Qubits':<15} {'Params'}")
print("-" * 40)
for gate in circuit_ir.gates:
    print(f"{gate.name:<12} {str(gate.qubits):<15} {gate.params}")

print(f"\nQubit connectivity: {circuit_ir.qubit_connectivity}")

## Summary

In this notebook you learned how to:

- Write an OpenQASM 2.0 circuit and normalize it with `CircuitNormalizer`
- Access the `CircuitIR` properties: `num_qubits`, `depth`, `gate_count`, `circuit_hash`
- Extract metadata with `extract_metadata()` for downstream pipeline decisions
- Inspect individual gate operations in the normalized representation

**Next:** [02_bell_state.ipynb](02_bell_state.ipynb) explores multiple circuit types
and shows how `circuit_hash` provides deterministic identification.